# EA2 — Despliegue y gobierno de una infraestructura de datos en la nube

**Big Data (ISD-25)** · Ingeniería de Software y Datos · IU Digital de Antioquia

| | |
|---|---|
| **Grupo** | *70* |
| **Integrantes** | Isabela Cuartas Vence · Juan Camilo Gomez Murillo|
| **Caso de estudio** | *Wanderbricks* |
| **Fecha de entrega** | domingo 6 de septiembre |
| **🎥 Enlace al video** | *(pegar aquí — 6 a 9 minutos, mínimo 3 minutos por integrante)* |

> ⚠️ **Antes de entregar:** verificar que el enlace del video abra desde una cuenta distinta a la propia.
> Un enlace inaccesible se califica como no entregado.

---
## 1. Contexto y problema

*Qué necesidad de infraestructura plantea el caso y qué debe soportar el entorno.*

En la EA1 construimos una base de datos analítica para Wanderbricks sobre Databricks, 
con capas bronce y plata para usuarios, reservas, actualizaciones de reservas y países. 
Esa evidencia resolvió el *qué* (modelo de datos, transformaciones, consultas), pero 
dejó sin resolver el *dónde y bajo qué reglas* vive esa información: quién puede leer 
la capa plata, quién puede escribir en bronce, cómo se automatiza la actualización 
diaria de las tablas sin intervención manual, y qué pasaría si Wanderbricks tuviera 
que auditar quién accedió a los datos de un usuario específico.

Esta necesidad no es hipotética: un marketplace real como Wanderbricks tiene equipos 
con distintos niveles de confianza sobre los mismos datos. El equipo de analítica de 
negocio necesita consultar reservas y países para reportes, pero no debería poder 
alterar las tablas fuente. El equipo de ingeniería de datos necesita escribir en 
bronce y plata, pero no necesariamente tiene por qué gestionar permisos de otros 
usuarios. Un administrador necesita visibilidad total para auditoría y gobierno. Sin 
un entorno que separe estas responsabilidades mediante catálogos, esquemas y permisos 
explícitos (GRANT), cualquier persona con acceso al workspace podría, en teoría, 
modificar o eliminar las tablas plata que alimentan los reportes de cancelación por 
país que construimos en la EA1.

Por eso, el entorno que se despliega en esta evidencia debe soportar tres cosas 
además del almacenamiento: **gobierno de acceso** (permisos diferenciados por capa y 
por rol), **automatización** (que la actualización de bronce a plata no dependa de 
que alguien ejecute el notebook manualmente cada vez que llegan datos nuevos) y 
**trazabilidad** (poder responder, vía linaje, de dónde viene cada tabla y qué la 
transforma). Adicionalmente, como Databricks es una plataforma gestionada (PaaS), el 
entorno debe dejar explícito qué responsabilidades asume el proveedor (aprovisionamiento 
de clústeres, parcheo, disponibilidad del metastore) y cuáles siguen siendo nuestras 
(diseño de esquemas, políticas de acceso, definición de jobs), como base para la 
comparación con IaaS que se pide más adelante.

---
## 2. Descripción de los datos

*Qué va a vivir en esta infraestructura: volumen esperado, frecuencia de actualización
y quién la va a consumir.*

La infraestructura que se despliega en esta evidencia opera sobre el mismo catálogo 
construido en la EA1: `bigdata_grupo70.wanderbricks`, con las tablas bronce y plata 
derivadas de `samples.wanderbricks.{users, bookings, booking_updates, countries}`. 
El volumen que debe soportar el entorno es el siguiente:

| Tabla (capa) | Filas | Rol en la infraestructura |
|---|---:|---|
| `bronze_users` | 124,509 | Insumo crudo, solo lectura para la mayoría de roles |
| `bronze_bookings` | 72,247 | Insumo crudo |
| `bronze_booking_updates` | 83,068 | Insumo crudo, el de mayor tasa de crecimiento (más filas que reservas) |
| `bronze_countries` | 168 | Tabla de dimensión, cambia con muy poca frecuencia |
| `silver_bookings_estado_actual` | 72,246 | Tabla de consumo analítico principal |
| `silver_bookings_anidado` | 72,246 | Tabla de consumo con estructura anidada |

En términos de **volumen**, ninguna tabla individual supera los 125,000 registros, lo 
cual en Databricks Free Edition no representa presión de cómputo relevante; el 
volumen relevante para el diseño de infraestructura no es el tamaño actual, sino el 
**patrón de crecimiento**: `booking_updates` ya tiene más filas que `bookings` 
(83,068 vs. 72,247), lo que confirma que es una tabla de eventos que crece con cada 
interacción del usuario sobre una reserva existente, no solo con reservas nuevas. Esto 
importa para el diseño del Job de automatización: la ingesta de `booking_updates` debe 
tratarse como incremental (nuevos eventos que llegan constantemente), mientras que 
`countries` es prácticamente estática y no necesita reprocesarse con la misma 
frecuencia que las demás.

En cuanto a **frecuencia de actualización esperada**, se distinguen tres patrones 
distintos, relevantes para decidir cómo programar el Job de la sección 6:
- `bronze_countries`: actualización esporádica (nuevos países o cambios de continente son infrecuentes).
- `bronze_users`: actualización diaria (altas de nuevos usuarios).
- `bronze_bookings` y `bronze_booking_updates`: actualización de alta frecuencia, ya que cada acción de un usuario (crear, modificar o cancelar una reserva) genera una fila nueva.

En cuanto a **quién consume esta infraestructura**, se identifican tres perfiles, que 
son la base directa de la matriz de roles de la sección 4:
- **Analista de negocio**: consume las tablas plata (`silver_bookings_estado_actual`, `silver_bookings_anidado`) para generar los reportes que se construyeron en la EA1 (tasa de cancelación por país, volumen por continente). No necesita ni debería tener acceso de escritura sobre bronce ni plata.
- **Ingeniero de datos**: es quien ejecuta o mantiene el pipeline bronce → plata, por lo que necesita permisos de escritura sobre ambas capas, pero no necesariamente permisos de administración de otros usuarios.
- **Administrador**: gestiona catálogos, esquemas, permisos de los otros dos roles y supervisa el Job de automatización; requiere visibilidad y control total sobre el entorno.

Esta segmentación por rol y por capa es la que se traduce, en la sección 3, en la 
organización de catálogos/esquemas, y en la sección 4, en los `GRANT` ejecutados 
sobre objetos concretos de este mismo catálogo. 

---
## 3. Decisiones de diseño y justificación
### 3.1 Diagrama de la arquitectura

*Fuentes → ingesta → almacenamiento → procesamiento → consumo.
Señalar explícitamente qué capa administra el proveedor y cuál el equipo.
Insertar la imagen o usar un diagrama en texto.*

```
![Diagrama](https://dbc-ec405e34-ae71.cloud.databricks.com/editor/files/3728136304566769?o=7474658234647057)

```

### 3.2 Matriz de roles

*Qué puede hacer cada rol sobre cada capa en un entorno real.*

| Rol | Bronce | Plata | Oro |
|---|---|---|---|
| Analista | | | |
| Ingeniero de datos | | | |
| Administrador | | | |

### 3.3 Especificación del equivalente IaaS

*Se diseña, no se implementa. Máquinas y dimensionamiento, sistema operativo,
software a instalar, red, almacenamiento, y estimación del esfuerzo de puesta
en marcha y de operación.*

### 3.4 Comparación IaaS / PaaS / SaaS

| Criterio | IaaS | PaaS | SaaS |
|---|---|---|---|
| Control | | | |
| Tiempo hasta el primer resultado | | | |
| Esfuerzo operativo | | | |
| Costo | | | |
| Escalabilidad | | | |
| Gobierno | | | |

**Conclusión:** *cuál conviene a este caso y por qué.*

---
## 4. Implementación
### 4.1 Organización del entorno

In [0]:
# TODO: catálogos, esquemas y volúmenes organizados con criterio

### 4.2 Permisos

*Al menos dos sentencias GRANT con niveles distintos sobre objetos distintos.*

In [0]:
# GRANT 1 — a quién, qué y por qué:
# TODO

# GRANT 2 — a quién, qué y por qué:
# TODO

# display(spark.sql(f"SHOW GRANTS ON TABLE {TABLA}"))

### 4.3 Linaje

*Evidenciar el linaje de una tabla desde el Catalog Explorer. Insertar la captura.*

### 4.4 Automatización

*Un Job con al menos dos tareas encadenadas y una programación definida.
Insertar la captura de una ejecución exitosa e indicar el identificador del Job.*

In [0]:
# TODO: código de las tareas que ejecuta el Job

---
## 5. Resultados

---
## 6. Conclusiones

*Qué funcionó, qué no funcionó y qué harían distinto si empezaran de nuevo.
Las conclusiones deben derivarse de los resultados mostrados arriba, no de expectativas generales.*

---
## 7. Reparto del trabajo y uso de IA

| Integrante | De qué se encargó | Qué sustenta en el video |
|---|---|---|
| | | |
| | | |
| | | |

**Uso de asistentes de IA:** *(indicar en qué partes se usó el Databricks Assistant u otra
herramienta. Está permitido; lo que se evalúa es que cada integrante pueda explicar
cualquier línea del código en el video.)*

> Este reparto debe coincidir con lo que cada persona demuestra en el video y con el
> historial de commits del repositorio. Las tres fuentes se contrastan al calificar.

---
## 📹 Preguntas obligatorias de sustentación

Cada integrante responde estas tres preguntas en su intervención del video:

1. ¿Qué parte de esta arquitectura administra el proveedor y cuál administran ustedes?
2. Muestre un GRANT que ejecutó y explique a quién le está dando qué, y por qué.
3. Si tuvieran que montar esto sobre máquinas virtuales, ¿qué sería lo primero que se les complicaría?

---
## ✅ Antes de entregar

- [ ] El diagrama de arquitectura está incluido y descrito
- [ ] Los dos GRANT están ejecutados y el SHOW GRANTS muestra el resultado
- [ ] El Job tiene dos o más tareas, está programado y hay evidencia de ejecución
- [ ] La comparación IaaS/PaaS/SaaS termina en una conclusión, no en una tabla suelta
- [ ] El enlace del video está en la portada y abre desde otra cuenta
- [ ] Todo confirmado en /ea2 y el HTML subido a Canvas